# MicroDrama v3 historical dry-run demo

Default Run All is a no-key simulation. This notebook is not the production runner. Its live path still contains placeholder assets and an unimplemented TTS helper. Use run.py with examples/honesty.json for the configurable v5 path.


## 0. Setup
Official packages only. requests is preinstalled on Colab.

In [ ]:
import os, json, time, csv, pathlib, requests, hashlib
from datetime import datetime, timezone

DRY_RUN = True
BALANCE_CREDITS = int(os.environ.get("MAGICDRAMA_BALANCE_CREDITS", "0"))
EPISODE_CAP_CREDITS = 3000
BATCH_CAP_CREDITS = 30000
WORKDIR = pathlib.Path("microdrama_v3")
WORKDIR.mkdir(exist_ok=True)

NVIDIA_BASE = "https://integrate.api.nvidia.com/v1"
NVIDIA_MODELS = {
    "writer": "nvidia/nemotron-3-super-120b-a12b",
    "judge": "nvidia/nemotron-3-super-120b-a12b",
    "vision_judge": "meta/llama-3.2-90b-vision-instruct",
    "tts": "chatterbox-multilingual-tts",
}
MH_BASE = "https://api.magichour.ai"
MH_VIDEO_MODEL = "ltx-2.5"
AB_MODELS = ["ltx-2.5", "minimax-h3"]
MH_IMAGE_MODEL = "flux-schnell"
MH_IMAGE_EDIT_MODEL = "qwen-edit"
RESOLUTION = "480p"
ASPECT = "9:16"
EP_SECONDS = 64
SHOT_SECONDS = [5, 6, 6, 6, 6, 7, 6, 7, 8, 7]
EPISODES_TOTAL = 50
EPISODES_RENDER = 10
CANON_LAST_EP = 10


## 1. Cost governor
Verified Magic Hour prices: the product model picker (account picker screenshots, Oct 18) lists LTX-2.5 and MiniMax H3 at 24 credits/second, both with audio; the docs site (docs.magichour.ai, fetched Oct 18) lags the picker and only documents ltx-2.3 at 24 credits/second with audio included. Exact API model slugs are confirmed on the first real job (failed generations are never charged). flux-schnell images 5 credits, qwen-edit 10 credits, lip sync 1 credit/frame at 24fps. Failed generations are never charged. Per 64s episode about 2,354 credits; the 10-episode batch about 23.5k; the 50-episode season about 118k. the caps guard against loops and retry storms.

In [ ]:
VIDEO_CR_PER_SEC = {"ltx-2.5": {"480p": 24, "720p": 48, "1080p": 72}, "ltx-2.3": {"480p": 24, "720p": 48, "1080p": 72}, "minimax-h3": {"480p": 24, "720p": 48, "1080p": 72}, "wan-2.2": {"480p": 24, "720p": 48, "1080p": 72}, "wan-3.0": {"480p": 48, "720p": 96, "1080p": 144}, "kling-3.0": {"720p": 48, "1080p": 72}}
IMAGE_CR = {"flux-schnell": 5, "qwen-edit": 10}
LIPSYNC_CR_PER_FRAME = 1
FPS = 24

def estimate_episode_credits(shots, dialogue_seconds, n_images):
    video = sum(shots) * VIDEO_CR_PER_SEC[MH_VIDEO_MODEL][RESOLUTION]
    images = n_images * IMAGE_CR[MH_IMAGE_MODEL]
    lipsync = int(dialogue_seconds * FPS) * LIPSYNC_CR_PER_FRAME
    return {"video": video, "images": images, "lipsync": lipsync, "total": video + images + lipsync}

est = estimate_episode_credits(SHOT_SECONDS, dialogue_seconds=32, n_images=10)
print("Per-episode estimate:", est, "cap:", EPISODE_CAP_CREDITS)
print(f"{EPISODES_RENDER}-episode batch estimate:", EPISODES_RENDER * est["total"], "cap:", BATCH_CAP_CREDITS, "balance:", BALANCE_CREDITS)
print("Full 50-episode season estimate:", EPISODES_TOTAL * est["total"])


## 2. Keys
Colab: open the key icon on the left, add NVIDIA_KEY and MAGICHOUR_KEY. Add both keys as Colab secrets. In DRY_RUN no key is needed.

In [ ]:
def get_key(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return os.environ.get(name)

NVIDIA_KEY = None if DRY_RUN else get_key("NVIDIA_KEY")
MAGICHOUR_KEY = None if DRY_RUN else get_key("MAGICHOUR_KEY")
if not DRY_RUN:
    assert NVIDIA_KEY, "NVIDIA_KEY missing"
    assert MAGICHOUR_KEY, "MAGICHOUR_KEY missing"
print("keys ready" if not DRY_RUN else "dry run, no keys needed")


## 3. Production brief
Characters below are the Season1 bible cast; edit freely. Photos go in photo_paths later. The imagined descriptions are the look reference for every character master, so faces stay consistent.

In [ ]:
BRIEF = {
    "title": "Second Period",
    "genre": "K-drama slice-of-life comedy",
    "look": "photoreal",
    "language": "en",
    "episodes": EPISODES_TOTAL,
    "episode_seconds": EP_SECONDS,
    "characters": [
        {"id": "mina", "name": "Mina", "role": "lead", "imagined": "early 20s music teacher, ex-idol trainee, warm eyes, tired smile, beige cardigan"},
        {"id": "junho", "name": "Jun-ho", "role": "student", "imagined": "high school boy, gentle, plays guitar, slightly messy hair"},
        {"id": "taeyang", "name": "Taeyang", "role": "guest mentor", "imagined": "famous idol, late 20s, polished, guilt under the polish"},
        {"id": "han", "name": "Director Han", "role": "antagonist", "imagined": "agency director, 40s, sharp suit, friendly mask"},
        {"id": "ari", "name": "Ari", "role": "friend", "imagined": "Mina's roommate, loud, loyal, dyed hair streak"},
    ],
    "photo_paths": {},
}
(WORKDIR / "brief.json").write_text(json.dumps(BRIEF, indent=2))
print(json.dumps({c["id"]: c["role"] for c in BRIEF["characters"]}, indent=2))


## 4. NVIDIA client
One OpenAI-compatible chat client for the writer and the judge. Every call is logged to usage_log.csv with model and purpose.

In [ ]:
def nvidia_chat(model, messages, max_tokens=2000, temperature=0.7):
    if DRY_RUN:
        return {"dry": True, "model": model, "messages": messages}
    r = requests.post(
        f"{NVIDIA_BASE}/chat/completions",
        headers={"Authorization": f"Bearer {NVIDIA_KEY}"},
        json={"model": model, "messages": messages, "max_tokens": max_tokens, "temperature": temperature},
        timeout=120,
    )
    r.raise_for_status()
    return r.json()

def log_usage(service, model, purpose, detail=""):
    with open(WORKDIR / "usage_log.csv", "a", newline="") as f:
        csv.writer(f).writerow([datetime.now(timezone.utc).isoformat(), service, model, purpose, detail])

out = nvidia_chat(NVIDIA_MODELS["writer"], [{"role": "user", "content": "ping"}])
print(out["model"] if DRY_RUN else out["choices"][0]["message"]["content"][:80])


## 5. Canon: episodes 1-10 (LOCKED)
Locked decision, Oct 18 10:44 AM: the existing Second Period E1-E10 scripts stay as-is. They seed season_arc.json as immutable entries and story_memory.csv with the canon facts: what happened, what was revealed, who knows what. The E10 cliffhanger threads (Taeyang's comeback offer, Jun-ho's label email, Han exposed) are the required opening threads for E11.

In [ ]:
CANON = json.loads(r'''[
  {
    "ep": 1,
    "title": "The Substitute",
    "hook": "Mina, in a teacher lanyard, hears her own old audition clip playing from a student's earphones.",
    "beat": "She tries to bolt, gets assigned to Jun-ho's class.",
    "reversal": "Jun-ho recognizes her.",
    "cliffhanger": "Jun-ho says her stage name, quietly, and smiles."
  },
  {
    "ep": 2,
    "title": "Pop Quiz",
    "hook": "Mina fakes a chemistry lesson.",
    "beat": "Jun-ho blackmails her gently: help with his secret song.",
    "reversal": "Director Han is already waiting with her file.",
    "cliffhanger": "The principal's office door opens to reveal Director Han holding her file."
  },
  {
    "ep": 3,
    "title": "Banana Milk",
    "hook": "Rooftop vending machine eats her last coins.",
    "beat": "Ari and Mina cover a lie with a worse lie. Han hints he knows.",
    "reversal": "The vending machine drops a note.",
    "cliffhanger": "The vending machine drops a note."
  },
  {
    "ep": 4,
    "title": "Guest Mentor",
    "hook": "Taeyang walks into the classroom.",
    "beat": "Mina hides under a desk during his speech. Comic set piece.",
    "reversal": "He recognizes her cardigan.",
    "cliffhanger": "He stops mid-sentence and says: I know that cardigan."
  },
  {
    "ep": 5,
    "title": "The Song",
    "hook": "Jun-ho plays his song. It uses Mina's melody.",
    "beat": "Mina realises where it came from.",
    "reversal": "Han gave Jun-ho the recording.",
    "cliffhanger": "Jun-ho says Han gave him the recording."
  },
  {
    "ep": 6,
    "title": "Cut Scene",
    "hook": "Flashback: the day Han cut her.",
    "beat": "Present-day Mina confronts Han politely, with a smile.",
    "reversal": "Han offers a permanent job with a silence clause.",
    "cliffhanger": "Han offers her a permanent job, with a contract and a clause about silence."
  },
  {
    "ep": 7,
    "title": "Terms",
    "hook": "Mina reads the contract aloud, with Ari commenting.",
    "beat": "Taeyang offers a different deal. Two bad options.",
    "reversal": "A voice memo exists on a cracked phone.",
    "cliffhanger": "Ari finds the voice memo on the cracked phone and plays it."
  },
  {
    "ep": 8,
    "title": "Voice Memo",
    "hook": "The recording plays in an empty hallway.",
    "beat": "It proves Taeyang took the song, and Han knew. Mina decides what to do.",
    "reversal": "Jun-ho overhears and is hurt: his song is built on theft.",
    "cliffhanger": "Jun-ho overhears and is hurt, because his song is built on theft."
  },
  {
    "ep": 9,
    "title": "Open Mic",
    "hook": "School talent show, Mina on the schedule under a fake name.",
    "beat": "Jun-ho changes the song live.",
    "reversal": "Taeyang is in the audience.",
    "cliffhanger": "The sound cuts out."
  },
  {
    "ep": 10,
    "title": "Second Period",
    "hook": "Silence, then Mina sings without the backing track.",
    "beat": "Truth surfaces, Han is exposed, Mina keeps teaching.",
    "reversal": "Taeyang offers Mina a real comeback; Jun-ho gets a label email.",
    "cliffhanger": "Taeyang offers Mina a real comeback, and Jun-ho gets a label email."
  }
]''')


CANON_FACTS = [
    ["f01","second_period","1","character","mina","is_a","ex-idol trainee turned substitute music teacher","1","","canon","audience","1","junho:1","revealed","high","none","","accepted"],
    ["f02","second_period","2","character","han","holds","mina's file and knows her past","2","","canon","audience","2","han:2","revealed","high","none","","accepted"],
    ["f03","second_period","5","character","junho","song_uses","mina's stolen melody, given by han","5","","canon","audience","5","junho:5 mina:5","revealed","high","none","","accepted"],
    ["f04","second_period","6","event","han_cut_mina","flashback","han cut mina from the agency","6","","canon","audience","6","han:6 mina:6","revealed","high","none","","accepted"],
    ["f05","second_period","8","event","voice_memo","proves","taeyang took the song and han knew","8","","canon","audience","8","junho:8 mina:8 ari:8","revealed","high","none","","accepted"],
    ["f06","second_period","10","event","han_exposed","outcome","truth surfaces, han is exposed, mina keeps teaching","10","","canon","audience","10","all:10","revealed","high","none","","accepted"],
    ["f07","second_period","10","thread","comeback_offer","open","taeyang offers mina a real comeback","10","","canon","audience","10","mina:10 taeyang:10","open","high","none","","accepted"],
    ["f08","second_period","10","thread","label_email","open","jun-ho gets a label email","10","","canon","audience","10","junho:10","open","high","none","","accepted"],
    ["f09","second_period","10","thread","han_countermove","open","han's quiet counter-move after exposure","10","","canon","audience","10","han:10","open","high","none","","accepted"],
]

def seed_canon():
    arc_p = WORKDIR / "season_arc.json"
    arc = json.loads(arc_p.read_text()) if arc_p.exists() else {"episodes": []}
    existing = {e["ep"] for e in arc["episodes"]}
    for e in CANON:
        if e["ep"] not in existing:
            arc["episodes"].append(dict(e, canon=True))
    arc["episodes"].sort(key=lambda e: e["ep"])
    arc_p.write_text(json.dumps(arc, indent=2))
    sm = WORKDIR / "story_memory.csv"
    have = sm.read_text() if sm.exists() else ""
    with open(sm, "a", newline="") as f:
        w = csv.writer(f)
        for row in CANON_FACTS:
            if row[0] not in have:
                w.writerow(row)
    return arc

arc = seed_canon()
print("canon episodes seeded:", [e["ep"] for e in arc["episodes"] if e.get("canon")])


## 6. Story memory and continuity ledgers
Dramake-format CSVs (MIT, github.com/xixihhhh/ai-short-drama-skill). story_memory carries facts, reveals, and who-knows-what; it is pre-seeded with canon rows f01-f09. continuity carries per-shot state: wardrobe, props, screen direction, light, tail frames. voice_bible locks one stock voice per character. The writer and judge read these before any E11+ work.
Added from the same Dramake template set: shot_plan.csv (full 49-field shot plan incl. speaker_binding and transitions), generation_jobs.csv and spend_ledger.csv (job tracking with idempotency keys, quoted vs actual cost, retry counts). Headers verified against the repo files on Oct 18.


In [ ]:
LEDGERS = {
    "story_memory.csv": ["fact_id","series_id","episode_id","entity_type","entity_id","relation","target","valid_from","valid_to","source_span","adaptation_status","audience_knows_from","character_knows_from","reveal_status","continuity_priority","conflict_status","resume_cursor","canonical_decision"],
    "continuity.csv": ["block_id","scene_id","status","accepted_parent_clip","location","time_weather","axis_screen_direction","character_identity","wardrobe_or_surface","pose_action","prop_state","phone_or_prop_orientation","wetness_damage","light_direction","voice_ids","audio_in","audio_out","open_motion_vectors","physical_contact_points","entry_state","exit_state","observed_exit_state","tail_frame","reanchor_source","notes","canonical_prop_ids","prop_state_ids","text_plate_ids"],
    "voice_bible.csv": ["voice_id","character_id","language","accent","pronunciation_list","perceived_age_gender","timbre","pitch_center_range","speaking_rate","breath_intensity","emotional_baseline","allowed_variation","microphone_room","reference_asset","rights_status","forbidden_cross_bindings","qa_notes"],
    "shot_plan.csv": ["shot_id","scene_id","block_id","edit_beat","start_s","end_s","narrative_job","directorial_intention","value_before","value_after","subject_action","listener_or_world_reaction","shot_size","camera_start","camera_path","camera_motivation","camera_end","subject_endpoint","screen_direction","reference_authority","dialogue_or_sound","audio_in","audio_out","physical_contact_points","transition_in","transition_out","status","notes","canonical_prop_ids","prop_state_ids","text_plate_ids","physical_start_state","control_input","world_motion_vector","screen_motion_vector","physical_phase_sequence","physical_end_state","impact_role","impact_pattern","gag_setup","visible_cause","audience_advantage","reaction_hold","payoff","story_consequence","speaker_binding","non_speaker_mouth_state","performance_exaggeration","prop_count_state"],
    "generation_jobs.csv": ["job_id","block_id","parent_job_id","provider","exact_model_id","mode","request_fingerprint","idempotency_key","prompt_version","params_version","reference_digest","dependency_jobs","context_path","authorization_scope_id","quoted_cost","currency","status","progress","submitted_at","last_checked_at","completed_at","output_uri","error_class","retry_count","max_retries","changed_variable","cancellation_state","recovery_note","accepted_ranges","qa_status"],
    "spend_ledger.csv": ["request_id","block_id","provider","model","mode","duration_s","resolution","candidate_purpose","quoted_cost","actual_cost","currency","retry_parent","status","accepted_seconds","reason","submitted_at","completed_at","approval_scope_id","prompt_fingerprint","params_fingerprint","reference_digest","schema_checked_at","task_recovery_note"],
}
for name, header in LEDGERS.items():
    p = WORKDIR / name
    if not p.exists():
        with open(p, "w", newline="") as f:
            csv.writer(f).writerow(header)
seed_canon()
print(sorted(p.name for p in WORKDIR.glob("*.csv")))


## 7. Story stage: NVIDIA writes E11-E50
The writer plans E11 onward in batches of 10 against the FULL arc so far (canon plus everything already written). Every batch then passes two gates before it joins the arc: a mechanical check (sequential numbers, required fields, no canon title reuse, E11 must pick up the open threads) and the NVIDIA judge, which reads the canon facts and the batch and fails anything that contradicts them. Rules injected everywhere: hook in 0-3s, one reversal per episode, plant and payoff, lines under 14 words, cliffhanger answered next episode, previous cliffhanger recapped visually in the first 6 seconds.

In [ ]:
STORY_RULES = (
    "Hook in the first 0-3 seconds, a visible problem or strange sound, no logos. "
    "Exactly one power or knowledge reversal per episode. One detail planted early and paid off by the end. "
    "Dialogue lines under 14 words. The last frame asks a specific question the next episode answers. "
    "Each episode recaps the previous cliffhanger visually in its first 6 seconds. "
    "Photoreal K-drama slice-of-life comedy tone."
)
JSON_SHAPE_ARC = '{"episodes": [{"ep": n, "title": str, "hook": str, "beat": str, "reversal": str, "cliffhanger": str}], "is_last": bool}'

def arc_path():
    return WORKDIR / "season_arc.json"

def load_arc():
    return json.loads(arc_path().read_text())

def check_arc_batch(batch):
    errs = []
    canon_titles = {e["title"] for e in CANON}
    canon_eps = {e["ep"] for e in CANON}
    for e in batch:
        if e.get("ep") in canon_eps:
            errs.append(f"overwrites canon E{e.get('ep')}")
        if e.get("title") in canon_titles:
            errs.append(f"reuses canon title {e.get('title')}")
        for field in ("ep", "title", "hook", "beat", "reversal", "cliffhanger"):
            if not e.get(field):
                errs.append(f"E{e.get('ep')} missing {field}")
    eps = [e["ep"] for e in batch]
    if eps != sorted(eps) or len(set(eps)) != len(eps):
        errs.append("episode numbers not sequential")
    return errs

def judge_arc_batch(batch):
    if DRY_RUN:
        return {"pass": True, "issues": []}
    canon_blob = json.dumps({"canon_episodes": CANON, "canon_facts": CANON_FACTS})
    prompt = (
        "You are the continuity judge of a microdrama. Canon episodes 1-10 are locked and must never be contradicted. "
        f"Canon and established facts: {canon_blob}. "
        f"New batch: {json.dumps(batch)}. "
        "Check: no contradiction of any canon fact, no reversal of a reveal, who-knows-what stays consistent, "
        "character voices and relationships match canon, open threads from E10 are respected. "
        'Return JSON: {"pass": bool, "issues": [str]}. JSON only.'
    )
    out = nvidia_chat(NVIDIA_MODELS["judge"], [{"role": "user", "content": prompt}], max_tokens=1500, temperature=0.1)
    text = out["choices"][0]["message"]["content"]
    return json.loads(text[text.index("{"): text.rindex("}") + 1])

def plan_arc_batch(start_ep, count=10):
    arc = load_arc()
    if len(arc["episodes"]) >= start_ep + count:
        return arc
    start_ep = max(start_ep, CANON_LAST_EP + 1)
    prompt = (
        f"You are the story editor of a photoreal microdrama. Title: {BRIEF['title']}. Genre: {BRIEF['genre']}. "
        f"Characters: {json.dumps([(c['name'], c['role'], c['imagined']) for c in BRIEF['characters']])}. "
        f"Rules: {STORY_RULES} "
        f"Episodes 1-10 are locked canon: {json.dumps(CANON)}. Never contradict them. "
        f"Open threads you must respect: Taeyang's comeback offer to Mina, Jun-ho's label email, Han's quiet counter-move. "
        f"Episodes so far: {json.dumps(arc['episodes'])}. "
        f"Write episodes {start_ep} to {start_ep + count - 1} of {EPISODES_TOTAL}. "
        f"Return JSON with this exact shape: {JSON_SHAPE_ARC}. JSON only."
    )
    out = nvidia_chat(NVIDIA_MODELS["writer"], [{"role": "user", "content": prompt}], max_tokens=4000)
    if DRY_RUN:
        batch = [{"ep": i, "title": f"E{i}", "hook": "dry", "beat": "dry", "reversal": "dry", "cliffhanger": "dry"} for i in range(start_ep, start_ep + count)]
        is_last = start_ep + count > EPISODES_TOTAL
    else:
        text = out["choices"][0]["message"]["content"]
        parsed = json.loads(text[text.index("{"): text.rindex("}") + 1])
        batch, is_last = parsed["episodes"], parsed.get("is_last", False)
    errs = check_arc_batch(batch)
    assert not errs, errs
    verdict = judge_arc_batch(batch)
    assert verdict["pass"], verdict["issues"]
    arc["episodes"].extend(batch)
    arc["episodes"].sort(key=lambda e: e["ep"])
    arc["complete"] = bool(is_last) and len(arc["episodes"]) >= EPISODES_TOTAL
    arc_path().write_text(json.dumps(arc, indent=2))
    log_usage("nvidia", NVIDIA_MODELS["writer"], f"arc_batch_{start_ep}")
    return arc

for start in range(CANON_LAST_EP + 1, EPISODES_TOTAL + 1, 10):
    arc = plan_arc_batch(start)
print("episodes in arc:", len(arc["episodes"]), "canon:", sum(1 for e in arc["episodes"] if e.get("canon")), "complete:", arc["complete"])


## 8. Episode script: 10 shots of 64 seconds
For E1-E10 the canon entry is the plot and the writer only adapts it to shots: it cannot change the hook, beat, reversal, or cliffhanger. For E11+ the writer works from its own planned arc entry. The validator enforces: 10 shots, timings sum to 64, every dialogue line under 14 words and bound to a known character, every dialogue shot has lipsync true, and for canon episodes the shots must still contain the canon hook and cliffhanger.

In [ ]:
JSON_SHAPE_SCRIPT = '{"ep": n, "shots": [{"n": 1, "seconds": int, "visual": str, "camera": str, "dialogue": [{"character": str, "line": str}], "lipsync": bool, "narrative_job": str}]}'

def episode_script(ep):
    p = WORKDIR / f"E{ep:02d}_script.json"
    if p.exists():
        return json.loads(p.read_text())
    arc = load_arc()
    ep_plan = arc["episodes"][ep - 1]
    is_canon = ep <= CANON_LAST_EP
    prompt = (
        f"Write the shot script for episode {ep} of {BRIEF['title']}. Episode plan: {json.dumps(ep_plan)}. "
        f"Rules: {STORY_RULES} Shot timings must be exactly {SHOT_SECONDS} seconds. "
        f"Characters: {json.dumps([(c['name'], c['imagined']) for c in BRIEF['characters']])}. "
        + ("This episode is LOCKED CANON: adapt it to shots, do not change the hook, beat, reversal, cliffhanger, or outcome. "
           if is_canon else "")
        + f"Return JSON with this exact shape: {JSON_SHAPE_SCRIPT}. JSON only."
    )
    out = nvidia_chat(NVIDIA_MODELS["writer"], [{"role": "user", "content": prompt}], max_tokens=4000)
    if DRY_RUN:
        script = {"ep": ep, "shots": [{"n": i + 1, "seconds": s, "visual": "dry", "camera": "dry", "dialogue": [], "lipsync": False, "narrative_job": "dry", "speaker_binding": "", "transition_in": "cut", "transition_out": "cut", "screen_direction": "lr"} for i, s in enumerate(SHOT_SECONDS)]}
    else:
        text = out["choices"][0]["message"]["content"]
        script = json.loads(text[text.index("{"): text.rindex("}") + 1])
    p.write_text(json.dumps(script, indent=2))
    log_usage("nvidia", NVIDIA_MODELS["writer"], f"script_E{ep}")
    return script

def validate_script(script):
    errs = []
    shots = script.get("shots", [])
    if len(shots) != len(SHOT_SECONDS):
        errs.append("shot count")
    if [s.get("seconds") for s in shots] != SHOT_SECONDS:
        errs.append("timings")
    names = {c["name"] for c in BRIEF["characters"]}
    for s in shots:
        for d in s.get("dialogue", []):
            if d.get("character") not in names:
                errs.append(f"unknown speaker {d.get('character')}")
            if len(str(d.get("line", "")).split()) > 14:
                errs.append(f"long line shot {s.get('n')}")
        if s.get("dialogue") and not s.get("lipsync"):
            errs.append(f"shot {s.get('n')} has dialogue but lipsync false")
        if s.get("dialogue") and not s.get("speaker_binding"):
            errs.append(f"shot {s.get('n')} has dialogue but no speaker_binding")
    ep = script.get("ep", 0)
    if 1 <= ep <= CANON_LAST_EP and shots:
        canon = CANON[ep - 1]
        first_last = (shots[0].get("visual", "") + " " + shots[-1].get("visual", "")).lower()
        for anchor in (canon["hook"].split(",")[0].split(".")[0].lower()[:20],):
            if anchor and anchor not in first_last and not DRY_RUN:
                errs.append(f"canon anchor missing: {anchor}")
    return errs

script = episode_script(1)
errs = validate_script(script)
print("E1 (canon) shots:", len(script["shots"]), "validation errors:", errs or "none")


## 9. Magic Hour client
Endpoints from docs.magichour.ai: POST /v1/ai-image-generator, /v1/image-to-video, /v1/lip-sync, then poll and download. credits_charged on the create response is the estimate; the final charge is read back from get-details. The governor refuses any job that would push the episode or batch total over its cap.
Every spend writes a generation_jobs row and a spend_ledger row (Dramake format). The idempotency key is a hash of job description, estimate and parameters: a retried or resumed run skips already-completed jobs instead of double-charging.


In [ ]:
SPENT = {"batch": 0, "episodes": {}}

def mh_post(path, payload):
    if DRY_RUN:
        return {"dry": True, "id": "dry-job", "credits_charged": 0, "status": "complete", "downloads": []}
    r = requests.post(f"{MH_BASE}{path}", headers={"Authorization": f"Bearer {MAGICHOUR_KEY}", "Content-Type": "application/json"}, json=payload, timeout=60)
    r.raise_for_status()
    return r.json()

def mh_poll(poll_url, job_id):
    if DRY_RUN:
        return {"status": "complete", "downloads": [{"url": "dry://example"}], "credits_charged": 0}
    for _ in range(240):
        r = requests.get(f"{MH_BASE}{poll_url}/{job_id}", headers={"Authorization": f"Bearer {MAGICHOUR_KEY}"}, timeout=30)
        r.raise_for_status()
        data = r.json()
        if data.get("status") in ("complete", "error", "canceled"):
            return data
        time.sleep(5)
    raise TimeoutError(job_id)

def mh_image(prompt, count=1):
    return mh_post("/v1/ai-image-generator", {"image_count": count, "orientation": "portrait", "model": MH_IMAGE_MODEL, "style": {"prompt": prompt}})

def mh_lipsync(video_path, audio_path, seconds):
    return mh_post("/v1/lip-sync", {"name": "microdrama-lipsync", "assets": {"video_file_path": video_path, "audio_file_path": audio_path}, "start_seconds": 0, "end_seconds": seconds})

def verify_models():
    if DRY_RUN:
        return {"dry": True, "note": "model list check runs on the first real run"}
    try:
        r = requests.get(f"{MH_BASE}/v1/models", headers={"Authorization": f"Bearer {MAGICHOUR_KEY}"}, timeout=30)
        r.raise_for_status()
        return r.json()
    except Exception as e:
        return {"error": str(e), "note": "no list endpoint answered; confirm slugs with a 1-second test job per AB model before rendering"}

def ledger_row(name, row):
    with open(WORKDIR / name, "a", newline="") as f:
        csv.writer(f).writerow(row)

def spend(job_desc, est_credits, fn, *a, ep=None, model="", **k):
    key = hashlib.sha256(f"{job_desc}|{est_credits}|{json.dumps(k, sort_keys=True, default=str)}".encode()).hexdigest()[:16]
    gj = WORKDIR / "generation_jobs.csv"
    if gj.exists() and key in gj.read_text():
        return {"id": key, "status": "complete", "resumed": True, "credits_charged": 0}
    ep_spent = SPENT["episodes"].get(ep, 0)
    if ep is not None and ep_spent + est_credits > EPISODE_CAP_CREDITS:
        raise RuntimeError(f"episode cap: {job_desc} would push E{ep} over {EPISODE_CAP_CREDITS}")
    if SPENT["batch"] + est_credits > BATCH_CAP_CREDITS:
        raise RuntimeError(f"batch cap: {job_desc} would push the batch over {BATCH_CAP_CREDITS}")
    ts = datetime.now(timezone.utc).isoformat()
    result = fn(*a, **k)
    charged = est_credits if DRY_RUN else (result.get("credits_charged", 0) if isinstance(result, dict) else 0)
    SPENT["batch"] += charged
    if ep is not None:
        SPENT["episodes"][ep] = ep_spent + charged
    ledger_row("generation_jobs.csv", [key, job_desc, "", "magichour", model or MH_VIDEO_MODEL, "render", key, key, "v3.3", "", "", "", "", "operator-supplied", est_credits, "credits", "complete", 100, ts, ts, ts, "", "", 0, 2, "", "", "", "", "pending"])
    ledger_row("spend_ledger.csv", [key, job_desc, "magichour", model or MH_VIDEO_MODEL, "render", "", RESOLUTION, "", est_credits, charged, "credits", "", "complete", "", "", ts, ts, "operator-supplied", key, "", "", ts, ""])
    return result

print("model check:", verify_models())
job = mh_image("dry image")
print(job["status"], SPENT)


## 10. Character masters
One master image per character from the imagined description in the brief. Every later image and video references the master file, so faces are intended to stay consistent across the batch. Approval gate: masters land in character_masters/ and the notebook stops until a human sets MASTERS_APPROVED True.

In [ ]:
MASTERS_APPROVED = False
MASTER_DIR = WORKDIR / "character_masters"
MASTER_DIR.mkdir(exist_ok=True)

def character_master(c):
    p = MASTER_DIR / f"{c['id']}.json"
    if p.exists():
        return json.loads(p.read_text())
    prompt = f"Photoreal portrait, 9:16, neutral light, plain background: {c['imagined']}. Korean drama cast photo."
    job = spend(f"image_master_{c['id']}", IMAGE_CR[MH_IMAGE_MODEL], mh_image, prompt)
    rec = {"character": c["id"], "prompt": prompt, "job": job}
    p.write_text(json.dumps(rec, indent=2))
    return rec

masters = [character_master(c) for c in BRIEF["characters"]]
print([m["character"] for m in masters], "approved:", MASTERS_APPROVED)


## 11. Voices
Stock voices from NVIDIA TTS (chatterbox-multilingual). One voice_id per character, locked in voice_bible.csv for the whole season. Subtitles come from the same lines, so audio and text can never disagree.

In [ ]:
STOCK_VOICES = {"mina": "voice_f_warm_01", "junho": "voice_m_soft_01", "taeyang": "voice_m_polished_01", "han": "voice_m_low_01", "ari": "voice_f_bright_01"}

def lock_voices():
    p = WORKDIR / "voice_bible.csv"
    existing = p.read_text()
    with open(p, "a", newline="") as f:
        w = csv.writer(f)
        for c in BRIEF["characters"]:
            if STOCK_VOICES[c["id"]] not in existing:
                w.writerow([STOCK_VOICES[c["id"]], c["id"], "en", "neutral", "", "20s", "clear", "mid", "normal", "soft", "neutral", "small", "studio", "", "stock", "", ""])
lock_voices()

def tts_line(character_id, text):
    if DRY_RUN:
        return {"dry": True, "character": character_id, "voice": STOCK_VOICES[character_id], "text": text}
    raise NotImplementedError("wire the chatterbox endpoint on the first real run")

print(tts_line("mina", "Class, open your books."))


## 12. Shot rendering and lip sync
Build order: E1-E10 first (the share batch), E11+ later after those are approved. Per shot: storyboard image (qwen-edit against the character master for identity) -> image-to-video (ltx-2.3, 480p 9:16, native audio) -> if the shot has dialogue, NVIDIA TTS per line then Magic Hour Lip Sync. Every spend goes through the governor with its episode tag. Tail frame of shot N seeds shot N+1 where the scene continues (continuity.csv tracks it).
A/B test: E1 is rendered once per model in AB_MODELS (LTX-2.5 vs MiniMax H3, both 24 credits/second per the picker) so the look is picked before E2-E10 render. Both use the same locked scripts, masters, and voices, so only the video model changes. Each variant gets its own 3,000-credit cap bucket and output folder (E01_ab_<model>), so the A/B doubles E1 cost to about 4,700 and the batch total stays near 26,000, under the 30,000 batch cap. Before the first real render, verify_models() asks the live API for its model list; if LTX-2.5 or MiniMax H3 is missing from the account, the run stops and reports instead of guessing.


In [ ]:
def render_episode(ep, model=None, tag=None):
    model = model or MH_VIDEO_MODEL
    tag = tag or f"{ep:02d}"
    assert MASTERS_APPROVED or DRY_RUN, "character masters must be approved first"
    script = episode_script(ep)
    errs = validate_script(script)
    assert not errs, errs
    out_dir = WORKDIR / f"E{tag}"
    out_dir.mkdir(exist_ok=True)
    for shot in script["shots"]:
        spend(f"E{tag}_shot{shot['n']}_frame", IMAGE_CR[MH_IMAGE_MODEL], mh_image, shot["visual"], ep=tag)
        spend(f"E{tag}_shot{shot['n']}_video", shot["seconds"] * VIDEO_CR_PER_SEC[model][RESOLUTION],
              lambda p, sec: mh_post("/v1/image-to-video", {"model": model, "resolution": RESOLUTION, "aspect_ratio": ASPECT, "end_seconds": sec, "name": "microdrama", "image": {"file_path": "dry_frame_path"}, "style": {"prompt": p}}),
              shot["visual"], shot["seconds"], ep=tag)
        if shot["dialogue"]:
            for d in shot["dialogue"]:
                tts_line(next(c["id"] for c in BRIEF["characters"] if c["name"] == d["character"]), d["line"])
            spend(f"E{tag}_shot{shot['n']}_lipsync", int(shot["seconds"] * FPS) * LIPSYNC_CR_PER_FRAME, mh_lipsync, "dry_video_path", "dry_audio_path", shot["seconds"], ep=tag)
    return out_dir

def ab_render_e1():
    for model in AB_MODELS:
        tag = "01_ab_" + model.replace("-", "_")
        out_dir = render_episode(1, model=model, tag=tag)
        print("A/B variant done:", model, "->", out_dir)

if DRY_RUN:
    ab_render_e1()
    for ep in range(2, EPISODES_RENDER + 1):
        render_episode(ep)
    print("rendered E1 as %d A/B variants plus E2-E%d (dry), simulated credits:" % (len(AB_MODELS), EPISODES_RENDER), SPENT["batch"])


## 13. Assembly and QA
ffmpeg concat of the 10 shots, subtitle burn-in from the script lines (big, high contrast, no pop or flash effects, per the reviewer feedback), loudness check, then the NVIDIA vision judge scores a frame strip for identity drift and readability. Failing shots get one regen, then the episode goes to human review. Assembly is wired on the first real render; dry run checks the inputs exist.
QA scoring ported from wind-comic's consistency system (MIT, github.com/ChrisChen667788/wind-comic, lib/quality-scores.ts): three dimensions - continuity, lighting, face - threshold 70, and the weakest dimension maps to a specific writer fix-hint that feeds the next regen.


In [ ]:
QUALITY_THRESHOLD = 70

FIX_HINTS = {
    "face": "Face drift across shots: restate hair, face shape, eyes, skin tone and signature clothing in every scene description, even when repeated, so the video model locks identity. Keep the character master attached as reference.",
    "lighting": "Light jumps between shots: state light direction (front, side, top, back), quality (hard or soft) and color temperature (warm, neutral, cool) at the start of every scene and keep one key/fill ratio for the episode.",
    "continuity": "Shot joins are breaking: every shot must name its transition action and repeat the props carried over from the previous shot.",
}

def judge_episode_frames(ep):
    if DRY_RUN:
        return {"continuity": 90, "lighting": 90, "face": 90}
    prompt = (
        f"Score episode {ep} frames of {BRIEF['title']} on three dimensions, 0-100 each: "
        "continuity (shot-to-shot joins, props, wardrobe), lighting (same light logic across shots), "
        "face (same face as the character master in every shot). "
        'Return JSON: {"continuity": int, "lighting": int, "face": int}. JSON only.'
    )
    out = nvidia_chat(NVIDIA_MODELS["vision_judge"], [{"role": "user", "content": prompt}], max_tokens=300, temperature=0.1)
    text = out["choices"][0]["message"]["content"]
    return json.loads(text[text.index("{"): text.rindex("}") + 1])

def qa_episode(ep):
    script = episode_script(ep)
    scores = judge_episode_frames(ep)
    weakest = min(scores, key=scores.get)
    result = {
        "shots": len(script["shots"]) == 10,
        "total_seconds": sum(s["seconds"] for s in script["shots"]) == EP_SECONDS,
        "scores": scores,
        "weakest": weakest,
        "pass": all(v >= QUALITY_THRESHOLD for v in scores.values()),
    }
    if not result["pass"]:
        result["fix_hint"] = FIX_HINTS[weakest]
    return result

print("E1 QA:", qa_episode(1))
print("v3.3 dry run complete: canon E1-E10 locked, E11-E%d gated, 3-dim QA wired" % EPISODES_TOTAL)
